# Lab 11: Card Fraud and the Accuracy Trap
Course 314341 AI in Cybersecurity and Cloud Computing, Ajloun National University. Dr. Mohammed Tawfik.

Run the cells from top to bottom in Google Colab (Runtime → Run all, or Shift+Enter cell by cell). Read the comments: lines that start with # explain the code.

**Data:** `creditcard.csv` from Kaggle [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud): 284,807 payments, only 492 frauds. The file is about 150 MB: upload it to Colab with the **Files** panel (or to Google Drive). Columns V1–V28 are anonymised by the bank; `Class` is 1 for fraud, 0 for normal.

## 1. Load and see how rare fraud is

In [ ]:
import pandas as pd
data = pd.read_csv("creditcard.csv")
print(data.shape)
print(data["Class"].value_counts())
X = data.drop(columns=["Class"])
y = data["Class"]
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

## 2. The lazy model: always says "normal"

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
lazy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
p = lazy.predict(X_test)
print("accuracy:", round(accuracy_score(y_test, p), 4), "| frauds caught:", recall_score(y_test, p))

## 3. A real model (numbers scaled first, because Time and Amount are large)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
p = model.predict(X_test)
print(confusion_matrix(y_test, p))
print("detection rate (recall):", round(recall_score(y_test, p), 3), "| precision:", round(precision_score(y_test, p), 3))

## 4. Give the rare frauds more weight

In [ ]:
model_w = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
model_w.fit(X_train, y_train)
p = model_w.predict(X_test)
print(confusion_matrix(y_test, p))
print("detection rate (recall):", round(recall_score(y_test, p), 3), "| precision:", round(precision_score(y_test, p), 3))

## 5. Move the alarm limit

In [ ]:
scores = model.predict_proba(X_test)[:, 1]                # chance of fraud
for limit in [0.1, 0.3, 0.5, 0.7, 0.9]:
    alarm = scores >= limit
    caught = (alarm & (y_test == 1)).sum()
    false_alarms = (alarm & (y_test == 0)).sum()
    print(f"limit {limit}: caught {caught} of {(y_test == 1).sum()}, false alarms {false_alarms}")